# Arabic verb sense matching: dataset overview

This notebook presents the datasets created for Arabic Word Sense Disambiguation experiments. It introduces the task through real annotated examples, then compares the composition, label balance and quality indicators of the two dataset versions.


## The task

The experiments formulate Arabic Word Sense Disambiguation as a binary matching problem:

> Given an Arabic verb in context and a candidate dictionary gloss, does the gloss express the intended meaning of the verb?

Each row contains:

- `verb_context`: the target verb together with its sentence context;
- `sense`: one candidate dictionary gloss;
- `label`: `1` for a matching pair and `0` for a non-matching pair.

An **instance** therefore means one context–gloss pair, not one unique verb or one unique sentence.


## Dataset versions

| Version | Instances | Experimental use |
|---|---:|---|
| Initial dataset | 51,856 | Stop-word and particle experiments |
| Revised dataset | 51,161 | Individual and ensemble BERT experiments |

These are two successive versions of the same research dataset. They are not training and test partitions.


## Setup

Both Excel files are included in the repository's `data` directory, so the analysis can be reproduced directly after installing the project dependencies.


In [1]:
from pathlib import Path

import pandas as pd

from arabic_wsd.data_quality import prepare_pairs, summarize_dataset

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = REPO_ROOT / "data"
INITIAL_FILE = "arabic_verb_context_gloss_pairs_initial.xlsx"
REVISED_FILE = "arabic_verb_context_gloss_pairs_revised.xlsx"


## 1. Dataset composition

The first comparison describes the overall scale of each version. Text values are normalized for spacing before unique contexts and senses are counted.


In [2]:
initial = pd.read_excel(DATA_DIR / INITIAL_FILE)
revised = pd.read_excel(DATA_DIR / REVISED_FILE)

initial_summary = summarize_dataset(initial).to_dict()
revised_summary = summarize_dataset(revised).to_dict()

summary = pd.DataFrame([
    {"Dataset": "Initial", **initial_summary},
    {"Dataset": "Revised", **revised_summary},
]).set_index("Dataset")

summary[[
    "instances", "positive_labels", "negative_labels",
    "unique_contexts", "unique_senses"
]].rename(columns={
    "instances": "Instances",
    "positive_labels": "Positive",
    "negative_labels": "Negative",
    "unique_contexts": "Unique contexts",
    "unique_senses": "Unique senses",
})


         Instances  Positive  Negative  Unique contexts  Unique senses
Dataset                                                               
Initial      51856     26091     25765            25203          21907
Revised      51161     25721     25440            25123          21751

The revised version remains close in size to the initial one. It contains **51,161 context–gloss pairs**, with more than **25,000 distinct normalized contexts** and **21,751 distinct normalized senses**.


## 2. Annotated examples

The following rows come from the revised dataset. Each context is paired with several candidate glosses. A positive label marks the gloss that matches the intended meaning in that context.


In [3]:
examples = prepare_pairs(revised).head(6).rename(columns={
    "verb_context": "Verb and context",
    "sense": "Candidate gloss",
    "label": "Label",
})

examples["Label"] = examples["Label"].map({1: "Match", 0: "Non-match"})
examples


,Verb and context,Candidate gloss,Label
0,"ائتكل,ائتكلت الديدان",اكلت بعضها بعضا,Match
1,"ائتكل,ائتكلت الديدان",اشتد لهيبها,Non-match
2,"ائتكل,ائتكلت النار",اكلت بعضها بعضا,Non-match
3,"ائتكل,ائتكلت النار",اشتد لهيبها,Match
4,"ائتكل,ائتكل الانسان غضبا",اشتد لهيبها,Non-match
5,"ائتكل,ائتكل الانسان غضبا",احترق,Match


The examples show why the task cannot be solved from the verb alone. The verb **ائتكل** appears with different meanings depending on its context: the gloss that matches **ائتكلت الديدان** does not match **ائتكلت النار**, and conversely.


## 3. Label balance

The positive and negative classes should remain comparable in size so that accuracy is not dominated by one class.


In [3]:
label_balance = pd.DataFrame({
    "Initial": [
        initial_summary["positive_labels"] / initial_summary["instances"],
        initial_summary["negative_labels"] / initial_summary["instances"],
    ],
    "Revised": [
        revised_summary["positive_labels"] / revised_summary["instances"],
        revised_summary["negative_labels"] / revised_summary["instances"],
    ],
}, index=["Positive", "Negative"])

label_balance.map(lambda value: f"{value:.2%}")


          Initial Revised
Positive   50.31%  50.27%
Negative   49.69%  49.73%

Both versions are almost evenly balanced. In the revised dataset, positive pairs account for **50.27%** of the instances and negative pairs for **49.73%**.


## 4. Data-quality indicators

Two indicators are computed after trimming whitespace, collapsing repeated spaces and standardizing spaces around commas:

- **Duplicate row:** the same normalized context, gloss and label occurs more than once.
- **Conflicting pair:** the same normalized context and gloss is associated with both labels.

These indicators identify structural inconsistencies; they do not measure linguistic annotation quality by themselves.


In [4]:
quality = summary[["duplicate_rows", "conflicting_pairs"]].rename(columns={
    "duplicate_rows": "Duplicate rows after normalization",
    "conflicting_pairs": "Conflicting context-gloss pairs",
})

quality.T


                                    Initial  Revised
Duplicate rows after normalization      182       44
Conflicting context-gloss pairs         302       26

The revision reduces both indicators substantially: normalized duplicate rows decrease from **182 to 44**, while conflicting context–gloss pairs decrease from **302 to 26**.


## 5. Aggregate difference between versions

The following figures compare the dataset-level totals. The reduction in instances is a net difference between the two versions; it is not a row-by-row history of the revision process.


In [5]:
instance_reduction = initial_summary["instances"] - revised_summary["instances"]
duplicate_reduction = 1 - revised_summary["duplicate_rows"] / initial_summary["duplicate_rows"]
conflict_reduction = 1 - revised_summary["conflicting_pairs"] / initial_summary["conflicting_pairs"]

pd.Series({
    "Net reduction in instances": instance_reduction,
    "Net reduction (%)": f"{instance_reduction / initial_summary['instances']:.4%}",
    "Duplicate-row reduction": f"{duplicate_reduction:.2%}",
    "Conflicting-pair reduction": f"{conflict_reduction:.2%}",
})


Net reduction in instances      695
Net reduction (%)               1.3402%
Duplicate-row reduction      75.82%
Conflicting-pair reduction   91.39%
dtype: object

## Conclusion

The revised dataset preserves the original class balance and most of its scale while reducing the two structural issues measured here. It provides the cleaner basis used for the later BERT and ensemble experiments.

This overview deliberately reports aggregate statistics only. Detailed linguistic validation and model performance are documented separately.
